In [ ]:
from glob import glob
from os import path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyBigWig
import seaborn as sns
import sys
from pathlib import Path

# Add project root to PYTHONPATH
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

from script import process_targets

In [ ]:
genes = process_targets.load_genes()
promoter_regions = process_targets.get_tree_constant_promoters(genes)
ctcf_regions = process_targets.get_ctcf_tree()

In [ ]:
## Load in the covariate data
covariate_glob = glob("../02_summary_JSONs/*.json")

covariate_data = []
for filepath in covariate_glob:
    collection_name = path.basename(filepath).split(".")[0]
    experiment_name = path.basename(filepath).split(".")[1]
    
    with open(filepath, "r", encoding="utf-8") as file:
        summary = json.load(file)

    summary["collection_name"] = collection_name
    summary["experiment_name"] = experiment_name
    
    covariate_data.append(summary)

print("Loading into dataframe")
covariate_data = pd.DataFrame(covariate_data)
covariate_columns = [col for col in covariate_data.columns if col not in ["collection_name", "experiment_name"]]

print(covariate_data.shape)
covariate_data.head()

In [ ]:
covariate_glob = glob("../03_targets/*.tsv")

target_data = pd.concat([pd.read_csv(x, sep="\t") for x in covariate_glob])

target_data.shape

In [ ]:
full_model_data = covariate_data.merge(target_data, on=["experiment_name", "collection_name"], )

full_model_data.shape

In [ ]:
def plot_promoters(
    results,
    total_datapoints,
    covariate="percentage_coverage",
    ranges=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0],
    labels=[
        ">90% background noise",
        "80%->90% background noise",
        "70%->80% background noise",
        "60%->70% background noise",
        "50%->60% background noise",
        "<50% background noise",
    ],
    title="Recovery of constant promoters sites for bigwigs split by percentage read background noise.",
):
    print(ranges)
    print(covariate)
    print(results.shape)
    for i in range(len(ranges) - 1):
        cond = list((full_model_data[covariate] < ranges[i + 1]) & (full_model_data[covariate] >= ranges[i]))
        sns.lineplot(
            x=np.array(list(range(3000))) - 1500,
            y=results[cond].mean(axis=0) / total_datapoints,
            label=labels[i]
        )

    plt.legend()
#     plt.xlim(-1500, 1500)
    plt.ylabel("Fraction of contstant promoter peaks called")
    plt.xlabel("Position relative to promoter peak centre")
    plt.title(title)
#     plt.savefig("01_plots/promoters_percentage_coverage.pdf", format="pdf", bbox_inches='tight', dpi=300)
#     plt.savefig("01_plots/promoters_percentage_coverage.png", format="png", bbox_inches='tight', dpi=300)
    plt.show()


def plot_ctcfs(
    results,
    total_datapoints,
    covariate="percentage_coverage",
    ranges=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0],
    labels=[
        ">90% background noise",
        "80%->90% background noise",
        "70%->80% background noise",
        "60%->70% background noise",
        "50%->60% background noise",
        "<50% background noise",
    ],
    title="Recovery of constant CTCF sites for bigwigs split by percentage read background noise.",
):
    for i in range(len(ranges) - 1):
        cond = list((full_model_data[covariate] < ranges[i + 1]) & (full_model_data[covariate] >= ranges[i]))
        sns.lineplot(
            x=np.array(list(range(3000))) - 1500,
            y=results[cond].mean(axis=0) / total_datapoints,
            label =labels[i]
        )

    plt.legend()
#     plt.xlim(-1500, 1500)
    plt.title(title)
    plt.ylabel("Fraction of contstant CTCF peaks called")
    plt.xlabel("Position relative to CTCF peak centre")
#     plt.savefig("01_plots/ctcfs_percentage_coverage.pdf", format="pdf", bbox_inches='tight', dpi=300)
#     plt.savefig("01_plots/ctcfs_percentage_coverage.png", format="png", bbox_inches='tight', dpi=300)
    plt.show()

In [ ]:
def get_promoter_medians(chrom):
    promoter_medians=[]
    tot_regions = len(promoter_regions[chrom])
    print(f"Working on {tot_regions} for {chrom}")
    for promoter in promoter_regions[chrom]:
        midpoint = (promoter.end + promoter.begin) // 2
        start=midpoint-1500
        end=midpoint+1500

        highest_coverage = []
        for _, row in full_model_data.iterrows():
            bigwig = pyBigWig.open(path.join(
                '/project/Wellcome_Discovery/datashare/CREST-GV_collection/',
                row.collection_name, "bigwigs",
                row.experiment_name + ".bw"
            ))
            raw_data = np.nan_to_num(
                np.array(bigwig.values(chrom, start, end)), nan=0.0
            )
            highest_coverage.append(np.argmax(raw_data))

        highest_coverage = np.array(highest_coverage)
        highest_coverage += start

        promoter_medians.append(np.median(highest_coverage))

        if len(promoter_medians) % 100 == 0:
            print(chrom, len(promoter_medians))
    return promoter_medians

total_datapoints = 0
for chrom in ["chr21", "chr1", "chr2", "chr3", "chr4", "chr5", "chr6"]:
    promoter_medians = get_promoter_medians(chrom)
    total_datapoints += len(promoter_medians)
    print(len(promoter_medians), total_datapoints)

plot_promoters(results, total_datapoints, title="")

In [ ]:
ctcf_results = np.zeros(shape=(full_model_data.shape[0], 3000), dtype=int)
total_datapoints = 0

completed = 0
for chrom in ["chr21", "chr1", "chr2", "chr3", "chr4", "chr5", "chr6"]:
    ctcf_intervals = ctcf_regions[chrom]
    total_datapoints += len(ctcf_intervals)
    print(f"Adding computation for {total_datapoints}")

    for ctcf_interval in ctcf_intervals:
        ctcf_midpoint = (ctcf_interval.end + ctcf_interval.begin) // 2
        start = int(ctcf_midpoint) - 1500
        end = int(ctcf_midpoint) + 1500
        length = 3000

        for i, row in full_model_data.iterrows():
            peak_file = pd.read_csv(
                path.join(
                    "/project/Wellcome_Discovery/datashare/CREST-GV_collection/peaks_unfiltered",
                    row.collection_name, row.experiment_name + "_L-tron.bed"
                ), sep="\t"
            )
            peak_file = peak_file[
                (peak_file.chrom == chrom) &
                (peak_file.start <= end) & (peak_file.end >= start)
            ]
            for _, row in peak_file.iterrows():
                pos1 = max(0, row.start - start)
                pos2 = min(length, row.end - start)
        #         print(pos1, pos2)

                ctcf_results[i, pos1:pos2] += 1

        if completed % 20 == 0:
            plot_ctcfs(ctcf_results, completed + 1, title="")

        completed += 1

plot_ctcfs(ctcf_results, completed, title="")